# ReformLab Advanced Notebook — Multi-Year Projections & Scenario Comparison

**Prerequisites:** Complete the [quickstart notebook](quickstart.ipynb) first to understand basic ReformLab concepts (policy objects, the adapter pattern, and the bridge to execution config).

**What you'll learn:**
1. Run multi-year projections with escalating policy rates
2. Configure vintage tracking with the `VintageTransitionStep`
3. Compare baseline vs. reform scenarios with distributional and fiscal indicators
4. **Build your own policy type** by subclassing `PolicyParameters`
5. Navigate run lineage graphs for reproducibility
6. Export results for external analysis

**Time:** ~40 minutes (reading + execution)

---
## Section 1: Multi-Year Escalating Policy

In the quickstart, we ran a single-year simulation. Now we'll project 10 years (2025-2034) with a carbon tax that increases over time — defined as a **typed policy object**.

In [ ]:
# Import the public API and typed policy objects
from pathlib import Path

from reformlab import (
    RunConfig,
    ScenarioConfig,
    create_quickstart_adapter,
    run_scenario,
    show,
)
from reformlab.templates.schema import (
    BaselineScenario,
    CarbonTaxParameters,
    FeebateParameters,
    PolicyParameters,
    PolicyType,
    YearSchedule,
)
from reformlab.vintage import (
    VintageCohort,
    VintageConfig,
    VintageState,
    VintageSummary,
    VintageTransitionRule,
    VintageTransitionStep,
)

# Resolve path relative to this notebook's location (works in both local and CI)
_NB_DIR = Path(__file__).parent if "__file__" in dir() else Path("notebooks")
POPULATION_PATH = _NB_DIR / "../data/populations/demo-quickstart-100.csv"

print("ReformLab API loaded successfully!")

### Define a 10-year escalating carbon tax policy

The carbon tax starts at €50/tCO2 in 2025 and rises to €100/tCO2 by 2030, then holds steady through 2034. We define this as a typed `CarbonTaxParameters` object.

In [ ]:
# 1. Define the typed policy with a 10-year rate schedule
reform_policy = CarbonTaxParameters(
    rate_schedule={
        2025: 50.0,   # €50/tCO2
        2026: 60.0,
        2027: 70.0,
        2028: 80.0,
        2029: 90.0,
        2030: 100.0,  # €100/tCO2 by 2030
        2031: 100.0,
        2032: 100.0,
        2033: 100.0,
        2034: 100.0,
    },
    redistribution_type="",
)

# 2. Wrap in a scenario
reform_scenario = BaselineScenario(
    name="france-carbon-tax-escalating",
    policy_type=PolicyType.CARBON_TAX,
    year_schedule=YearSchedule(start_year=2025, end_year=2034),
    policy=reform_policy,
    description="Escalating carbon tax: €50→€100/tCO2 over 2025-2034",
)

print(f"Policy: {type(reform_policy).__name__}")
print(f"Rate schedule: €50/tCO2 (2025) → €100/tCO2 (2030+)")
print(f"Scenario: {reform_scenario.name}")
print(f"Years: {reform_scenario.year_schedule.start_year}-{reform_scenario.year_schedule.end_year}")

In [ ]:
# 3. Bridge to execution config
import pyarrow.csv as pa_csv

population_table = pa_csv.read_csv(POPULATION_PATH)
num_households = population_table.num_rows

multi_year_config = RunConfig(
    scenario=ScenarioConfig(
        template_name=reform_scenario.name,
        policy={"rate_schedule": dict(reform_policy.rate_schedule)},
        population_path=POPULATION_PATH,
        start_year=reform_scenario.year_schedule.start_year,
        end_year=reform_scenario.year_schedule.end_year,
    ),
    seed=42,
)

# Note: The demo adapter applies a fixed carbon_tax_rate for all years.
# It does NOT read the rate_schedule per year — that's a production adapter feature.
# The rate_schedule is preserved in the manifest for reproducibility and governance.
adapter_multi = create_quickstart_adapter(
    carbon_tax_rate=50.0,
    year=2025,
)

print("Running 10-year simulation...")
result_multi = run_scenario(multi_year_config, adapter=adapter_multi)
print(result_multi)

### Inspect the multi-year panel output

The panel now contains 10 years × 100 households = 1,000 rows — all computed from the `CarbonTaxParameters` policy defined above.

In [ ]:
print(f"Panel shape: {result_multi.panel_output.table.num_rows} rows × {result_multi.panel_output.table.num_columns} columns")
print(f"Years represented: {sorted(result_multi.yearly_states.keys())}")
print()
show(result_multi.panel_output.table, n=15)

### Visualize yearly progression

Let's plot how the mean carbon tax burden evolves over the 10-year horizon.

In [ ]:
import matplotlib.pyplot as plt

# Plot yearly progression using the built-in plot method
fig, ax = result_multi.plot_yearly("carbon_tax")
plt.show()

print("\nInterpretation:")
print("- The demo adapter applies a fixed rate (€50/tCO2) across all years")
print("- In production, an OpenFisca adapter would read the rate_schedule per year,")
print("  producing an escalating burden curve matching the €50→€100/tCO2 schedule")
print("- The rate_schedule is still recorded in the manifest for governance")

**What just happened?**
- The `CarbonTaxParameters` policy defined the rate schedule (€50→€100/tCO2)
- The orchestrator executed 10 yearly loops (2025-2034)
- The demo adapter applied a fixed rate (€50/tCO2) for all years — in production, an OpenFisca adapter reads the rate per year from the schedule
- Results were stacked into a single panel dataset
- Year-by-year state is preserved in `result_multi.yearly_states`

---
## Section 2: Vintage Tracking

**Vintage tracking** models how household assets (vehicles, heating systems) age and get replaced over time. ReformLab's `VintageTransitionStep` is a real orchestrator step that plugs into the step pipeline alongside `ComputationStep`.

**How it works:**
- `VintageTransitionStep` implements the `OrchestratorStep` protocol
- Each year: ages all cohorts by +1, retires vehicles past `max_age`, adds new fleet entries at age 0
- Vintage state is stored in `YearState.data["vintage_vehicle"]` and carried between years
- You inject it into `run_scenario()` via the `steps=` parameter

We'll reuse the same `reform_policy` from Section 1 — the vintage step runs alongside the carbon tax computation.

In [ ]:
# Configure the vintage transition step for the vehicle fleet
vintage_config = VintageConfig(
    asset_class="vehicle",
    rules=(
        VintageTransitionRule(
            rule_type="fixed_entry",
            parameters={"count": 10},
            description="10 new vehicles enter the fleet each year",
        ),
        VintageTransitionRule(
            rule_type="max_age_retirement",
            parameters={"max_age": 15},
            description="Vehicles older than 15 years are retired",
        ),
    ),
    description="Vehicle fleet vintage tracking with fixed entry and retirement",
)
vintage_step = VintageTransitionStep(vintage_config)

print(f"Step name: {vintage_step.name}")
print(f"Asset class: {vintage_config.asset_class}")
print(f"Rules: {len(vintage_config.rules)} configured")
print(f"  Entry: {vintage_config.entry_rules[0].description}")
print(f"  Retirement: {vintage_config.retirement_rules[0].description}")

In [ ]:
# Define the initial fleet composition (55 vehicles spread across ages 0-10)
# Without initial_state, the fleet starts empty and grows from zero.
initial_fleet = VintageState(
    asset_class="vehicle",
    cohorts=tuple(
        VintageCohort(age=age, count=10 - age)
        for age in range(11)  # ages 0 through 10
    ),
)
print(f"Initial fleet: {initial_fleet.total_count} vehicles across {len(initial_fleet.cohorts)} cohorts")
print(f"Age distribution: {initial_fleet.age_distribution}")

# Re-run the multi-year simulation with the vintage step in the pipeline
result_vintage = run_scenario(
    multi_year_config,
    adapter=adapter_multi,
    steps=(vintage_step,),
    initial_state={"vintage_vehicle": initial_fleet},
)
print(result_vintage)

In [ ]:
# Extract vintage state from each year and display summaries
print("Vintage fleet evolution over 10 years:\n")
print(f"{'Year':<8} {'Total':<8} {'Cohorts':<10} {'Mean Age':<10} {'Max Age':<10}")
print("-" * 46)
for year in sorted(result_vintage.yearly_states.keys()):
    vintage_state = result_vintage.yearly_states[year].data["vintage_vehicle"]
    summary = VintageSummary.from_state(vintage_state)
    print(f"{year:<8} {summary.total_count:<8} {summary.cohort_count:<10} {summary.mean_age:<10.1f} {summary.max_age:<10}")

In [ ]:
from reformlab.visualization import create_figure

# Extract age distribution per year for stacked area chart
years = sorted(result_vintage.yearly_states.keys())
young_counts = []
mid_counts = []
old_counts = []
for year in years:
    vs = result_vintage.yearly_states[year].data["vintage_vehicle"]
    dist = vs.age_distribution
    young_counts.append(sum(dist.get(a, 0) for a in range(0, 6)))
    mid_counts.append(sum(dist.get(a, 0) for a in range(6, 11)))
    old_counts.append(sum(dist.get(a, 0) for a in range(11, 16)))

fig, ax = create_figure(
    title="Vehicle Fleet Vintage Evolution (2025-2034)",
    xlabel="Year",
    ylabel="Number of Vehicles",
)
ax.stackplot(
    years, young_counts, mid_counts, old_counts,
    labels=["Young (0-5 years)", "Mid-age (6-10 years)", "Old (11-15 years)"],
    colors=["#2ca02c", "#ff7f0e", "#d62728"],
    alpha=0.8,
)
ax.legend(loc="upper left", fontsize=10)
ax.set_xticks(years)
plt.show()

**What happened in the vintage pipeline:**
- The orchestrator ran `ComputationStep` (carbon tax from our `reform_policy`) then `VintageTransitionStep` for each of the 10 years
- Each year, all cohorts aged by +1, vehicles older than 15 were retired, and 10 new vehicles entered at age 0
- Every number in the chart above comes from `result_vintage.yearly_states[year].data["vintage_vehicle"].age_distribution`

**Why this matters:**
- The `steps=` parameter lets you compose custom pipelines — vintage is just one example
- Any step implementing the `OrchestratorStep` protocol can be plugged in
- Vintage state is fully deterministic: same config, same seed, same cohort counts

---
## Section 3: Baseline vs. Reform Comparison

Policy analysis requires comparing **baseline** (current policy) vs. **reform** (proposed policy) scenarios.

We'll compare:
1. **Baseline**: `CarbonTaxParameters(rate_schedule={y: 44.0 for y in range(2025, 2035)})` — constant €44/tCO2
2. **Reform**: The escalating `reform_policy` from Section 1 (€50→€100/tCO2)

Both run with vintage tracking for fair comparison.

### Define and run the baseline scenario

The baseline holds the carbon tax at €44/tCO2 — defined as a typed policy object.

In [ ]:
# 1. Define the baseline policy — constant €44/tCO2 for 10 years
baseline_policy = CarbonTaxParameters(
    rate_schedule={y: 44.0 for y in range(2025, 2035)},
    redistribution_type="",
)

# 2. Wrap in a scenario
baseline_scenario = BaselineScenario(
    name="france-carbon-tax-constant",
    policy_type=PolicyType.CARBON_TAX,
    year_schedule=YearSchedule(start_year=2025, end_year=2034),
    policy=baseline_policy,
    description="Constant carbon tax at €44/tCO2 (current rate)",
)

# 3. Bridge to execution config and run
baseline_config = RunConfig(
    scenario=ScenarioConfig(
        template_name=baseline_scenario.name,
        policy={"rate_schedule": dict(baseline_policy.rate_schedule)},
        population_path=POPULATION_PATH,
        start_year=baseline_scenario.year_schedule.start_year,
        end_year=baseline_scenario.year_schedule.end_year,
    ),
    seed=42,
)

adapter_baseline = create_quickstart_adapter(
    carbon_tax_rate=44.0,
    year=2025,
)

# Both baseline and reform run with vintage tracking for fair comparison
print("Running baseline scenario (€44/tCO2 constant) with vintage tracking...")
result_baseline = run_scenario(
    baseline_config,
    adapter=adapter_baseline,
    steps=(vintage_step,),
    initial_state={"vintage_vehicle": initial_fleet},
)
print(result_baseline)

### Compute distributional indicators for both scenarios

We'll compute mean carbon tax by income decile for baseline and reform.

In [ ]:
# Compute distributional indicators
indicators_baseline = result_baseline.indicators("distributional")
indicators_reform = result_vintage.indicators("distributional")

print(f"Baseline indicators: {len(indicators_baseline.indicators)} records")
print(f"Reform indicators: {len(indicators_reform.indicators)} records")

### Side-by-side comparison using ReformLab's comparison API

In [ ]:
from reformlab.indicators import (
    ComparisonConfig,
    FiscalConfig,
    ScenarioInput,
    compare_scenarios,
)

# Prepare scenario inputs
scenarios = [
    ScenarioInput(label="baseline", indicators=indicators_baseline),
    ScenarioInput(label="reform", indicators=indicators_reform),
]

# Compare with deltas
comparison = compare_scenarios(
    scenarios,
    config=ComparisonConfig(
        baseline_label="baseline",
        include_deltas=True,
        include_pct_deltas=True,
    ),
)

print(f"Comparison table: {comparison.table.num_rows} rows × {comparison.table.num_columns} columns")
print(f"Columns: {comparison.table.column_names}")
print()
show(comparison.table, n=20)

### Visualize baseline vs. reform impact by decile

In [ ]:
# Baseline vs Reform comparison using the built-in plot_comparison method
fig, ax = result_baseline.plot_comparison(result_vintage, "carbon_tax")
plt.show()

print("\nInterpretation:")
print("- Reform scenario increases burden across all deciles due to higher tax rate")
print("- Absolute burden increases with income (progressive in absolute terms)")
print("- To assess regressivity, compute burden as % of income (welfare indicators)")

### Compute per-year differences

The comparison table includes `delta_reform` (absolute difference) and `pct_delta_reform` (percentage change) columns.

In [ ]:
# Extract mean carbon tax delta across all deciles by year
import pyarrow.compute as pc

comp_table = comparison.table

if "delta_reform" in comp_table.column_names:
    carbon_tax_mean = comp_table.filter(
        pc.and_(
            pc.equal(comp_table["field_name"], "carbon_tax"),
            pc.equal(comp_table["metric"], "mean")
        )
    )

    if carbon_tax_mean.num_rows > 0:
        years_list = carbon_tax_mean["year"].to_pylist()
        years_delta = sorted([y for y in set(years_list) if y is not None])

        if years_delta:
            avg_delta_by_year = []
            for year in years_delta:
                year_data = carbon_tax_mean.filter(pc.equal(carbon_tax_mean["year"], year))
                avg_delta = pc.mean(year_data["delta_reform"]).as_py()
                avg_delta_by_year.append(avg_delta)

            fig, ax = create_figure(
                title="Reform Impact Over Time (Average Across Deciles)",
                xlabel="Year",
                ylabel="Mean Carbon Tax Difference (Reform - Baseline, €/year)",
            )
            ax.plot(years_delta, avg_delta_by_year, marker="o", linewidth=2, color="coral", markersize=8)
            ax.axhline(y=0, color="black", linestyle="--", linewidth=1, alpha=0.5)
            ax.set_xticks(years_delta)
            plt.show()
        else:
            print("No valid year data available for delta visualization")
    else:
        print("No carbon tax comparison data available")
else:
    print("Delta columns not available in comparison table")

### Compare fiscal indicators

We'll compare **fiscal indicators** for baseline and reform — treating `carbon_tax` as the revenue field.

In [ ]:
# Compute fiscal indicators for baseline and reform
fiscal_baseline = result_baseline.indicators(
    "fiscal",
    config=FiscalConfig(revenue_fields=["carbon_tax"], cost_fields=[]),
)
fiscal_reform = result_vintage.indicators(
    "fiscal",
    config=FiscalConfig(revenue_fields=["carbon_tax"], cost_fields=[]),
)

fiscal_comparison = compare_scenarios(
    [
        ScenarioInput(label="baseline", indicators=fiscal_baseline),
        ScenarioInput(label="reform", indicators=fiscal_reform),
    ],
    config=ComparisonConfig(
        baseline_label="baseline",
        include_deltas=True,
        include_pct_deltas=True,
    ),
)

print("Fiscal comparison table (excerpt):")
show(fiscal_comparison.table, n=20)

# Plot yearly fiscal revenue differences
fiscal_revenue = fiscal_comparison.table.filter(
    pc.equal(fiscal_comparison.table["metric"], "revenue")
).sort_by([("year", "ascending")])

if fiscal_revenue.num_rows > 0:
    fiscal_years = fiscal_revenue["year"].to_pylist()
    baseline_revenue = fiscal_revenue["baseline"].to_pylist()
    reform_revenue = fiscal_revenue["reform"].to_pylist()

    fig, ax = create_figure(
        title="Fiscal Comparison: Baseline vs Reform Revenue",
        xlabel="Year",
        ylabel="Carbon Tax Revenue (€/year)",
    )
    ax.plot(fiscal_years, baseline_revenue, marker="o", linewidth=2, label="Baseline revenue")
    ax.plot(fiscal_years, reform_revenue, marker="o", linewidth=2, label="Reform revenue")
    ax.legend(fontsize=11)
    ax.set_xticks(fiscal_years)
    plt.show()
else:
    print("No fiscal revenue rows available for plotting")

**What we've learned:**
- Both `baseline_policy` and `reform_policy` are typed `CarbonTaxParameters` objects
- ReformLab supports multi-scenario comparison with distributional and fiscal indicators
- Comparison includes absolute and percentage deltas
- Year-by-year comparison reveals how policy impacts evolve over time

---
## Section 4: Build Your Own Policy Type

All the built-in policies (`CarbonTaxParameters`, `SubsidyParameters`, `RebateParameters`, `FeebateParameters`) inherit from `PolicyParameters`. You can do the same to model any policy type.

The pattern is: **subclass → instantiate → wrap in scenario → run**.

> **Note:** The demo adapter used in this notebook applies a generic carbon tax formula regardless of the policy type. In production, you would write an OpenFisca adapter (or custom adapter) that reads your policy-specific parameters (`pivot_point`, `fee_rate`, etc.). The point of this section is that **the orchestrator pipeline doesn't care about the policy type** — any `PolicyParameters` subclass flows through the same execution path.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class SimpleFeebateParameters(PolicyParameters):
    """A feebate taxes high emitters and rebates low emitters.

    Households emitting above the pivot_point pay a fee;
    households below it receive a rebate.
    """
    pivot_point: float = 5.0       # tCO2/year threshold
    fee_rate: float = 50.0         # €/tCO2 above pivot
    rebate_rate: float = 30.0      # €/tCO2 below pivot

# Instantiate the custom policy
feebate_policy = SimpleFeebateParameters(
    rate_schedule={2025: 0.0},     # Base rate not used in feebate logic
    pivot_point=5.0,
    fee_rate=50.0,
    rebate_rate=30.0,
)

print(f"Custom policy type: {type(feebate_policy).__name__}")
print(f"Inherits from:      {type(feebate_policy).__bases__[0].__name__}")
print(f"Pivot point:        {feebate_policy.pivot_point} tCO2/year")
print(f"Fee rate:           €{feebate_policy.fee_rate}/tCO2 above pivot")
print(f"Rebate rate:        €{feebate_policy.rebate_rate}/tCO2 below pivot")

In [ ]:
# Wrap in a scenario and bridge to execution config
# Using the same 10-year horizon as Section 1 for a fair comparison
feebate_scenario = BaselineScenario(
    name="feebate-demo",
    policy_type=PolicyType.FEEBATE,
    year_schedule=YearSchedule(start_year=2025, end_year=2034),
    policy=feebate_policy,
    description="Simple feebate: tax high emitters, rebate low emitters",
)

feebate_config = RunConfig(
    scenario=ScenarioConfig(
        template_name=feebate_scenario.name,
        policy={
            "rate_schedule": dict(feebate_policy.rate_schedule),
            "pivot_point": feebate_policy.pivot_point,
            "fee_rate": feebate_policy.fee_rate,
            "rebate_rate": feebate_policy.rebate_rate,
        },
        population_path=POPULATION_PATH,
        start_year=feebate_scenario.year_schedule.start_year,
        end_year=feebate_scenario.year_schedule.end_year,
    ),
    seed=42,
)

# The demo adapter applies a generic carbon tax formula — in production,
# a feebate adapter would use pivot_point, fee_rate, and rebate_rate.
# The custom parameters are still recorded in the manifest for governance.
adapter_feebate = create_quickstart_adapter(
    carbon_tax_rate=44.0,
    year=2025,
)

print("Running feebate scenario...")
result_feebate = run_scenario(feebate_config, adapter=adapter_feebate)
print(result_feebate)

In [ ]:
# Compare feebate with the carbon tax from Section 1
# Both use 10-year horizons for a fair comparison
fig, ax = result_multi.plot_comparison(
    result_feebate,
    "carbon_tax",
    reform_label="Feebate (custom policy)",
)
plt.show()

print("\nThe custom SimpleFeebateParameters policy ran through the same pipeline")
print("as the built-in CarbonTaxParameters. The orchestrator treats all PolicyParameters")
print("subclasses identically — your custom fields are passed through to the adapter")
print("and recorded in the manifest. A production adapter would use them for computation.")

### The pattern

Creating a custom policy type is always the same:

1. **Subclass** `PolicyParameters` with your domain-specific fields
2. **Instantiate** with concrete values
3. **Wrap** in a `BaselineScenario`
4. **Bridge** to `ScenarioConfig` → `RunConfig`
5. **Run** with `run_scenario()` — the orchestrator doesn't care about the policy type

This works because `PolicyParameters` provides the base contract (`rate_schedule`, `exemptions`, etc.) and your subclass adds whatever fields your policy needs.

---
## Section 5: Lineage and Reproducibility

Every ReformLab simulation produces an **immutable run manifest** with complete provenance: parameter snapshots, seed values, data hashes, step pipeline, and lineage graphs (parent→child relationships for multi-year runs).

This enables audit trails, reproducibility verification, and governance compliance.

### Inspect the run manifest

In [ ]:
import json

# Access manifest from reform scenario (with vintage tracking)
manifest = result_vintage.manifest

print("=== Run Manifest (Reform Scenario) ===")
print(f"Manifest ID:      {manifest.manifest_id}")
print(f"Created at:       {manifest.created_at}")
print(f"Engine version:   {manifest.engine_version}")
print(f"Adapter version:  {manifest.adapter_version}")
print(f"Scenario version: {manifest.scenario_version}")
print(f"Policy:           {json.dumps(manifest.policy, indent=2)}")
print(f"Seeds:            {manifest.seeds}")
print(f"Warnings:         {manifest.warnings if manifest.warnings else 'None'}")
print(f"Step pipeline:    {manifest.step_pipeline}")

### Cross-scenario lineage (baseline and reform)

Each scenario run has its own manifest lineage. The `baseline_policy` and `reform_policy` objects produced distinct execution traces.

In [ ]:
# Inspect baseline vs reform lineage metadata
baseline_manifest = result_baseline.manifest
reform_manifest = result_vintage.manifest

print("Scenario-level manifests:")
print(f"  Baseline manifest: {baseline_manifest.manifest_id}")
print(f"  Reform manifest:   {reform_manifest.manifest_id}")
print()

for label, m in (("baseline", baseline_manifest), ("reform", reform_manifest)):
    print(f"{label.title()} lineage summary:")
    print(f"  Parent manifest ID: {m.parent_manifest_id or 'None (top-level run)'}")
    print(f"  Child manifests:    {len(m.child_manifests)} years")
    if m.child_manifests:
        first_year = min(m.child_manifests)
        last_year = max(m.child_manifests)
        print(f"  Horizon covered:    {first_year}-{last_year}")
    print()

if baseline_manifest.manifest_id != reform_manifest.manifest_id:
    print("Cross-scenario relationship:")
    print("  Baseline and reform are distinct top-level runs with separate lineages.")
    print("  Compare both manifest IDs in reports to preserve provenance across scenarios.")

### Verify deterministic rerun

If we rerun the exact same configuration with the same seed, we should get identical outputs.

In [ ]:
# Rerun with identical configuration (including vintage step and initial state)
print("Rerunning reform scenario with identical config...")
adapter_rerun = create_quickstart_adapter(
    carbon_tax_rate=50.0,
    year=2025,
)

result_rerun = run_scenario(
    multi_year_config,
    adapter=adapter_rerun,
    steps=(vintage_step,),
    initial_state={"vintage_vehicle": initial_fleet},
)
print(result_rerun)

In [ ]:
# Compare outputs (panel + vintage state)
original_panel = result_vintage.panel_output.table
rerun_panel = result_rerun.panel_output.table

print(f"Original panel: {original_panel.num_rows} rows")
print(f"Rerun panel:    {rerun_panel.num_rows} rows")

if original_panel.num_rows == rerun_panel.num_rows:
    original_tax = original_panel["carbon_tax"].to_pylist()
    rerun_tax = rerun_panel["carbon_tax"].to_pylist()

    if original_tax == rerun_tax:
        print("✓ Rerun matches original output exactly (deterministic!)")
    else:
        max_diff = max(abs(o - r) for o, r in zip(original_tax, rerun_tax))
        print(f"⚠ Small differences detected (max diff: {max_diff:.6f})")
else:
    print("✗ Row counts differ — non-deterministic behavior detected")

# Verify vintage state determinism
print("\nVintage state determinism check:")
all_match = True
for year in sorted(result_vintage.yearly_states.keys()):
    orig_vs = result_vintage.yearly_states[year].data["vintage_vehicle"]
    rerun_vs = result_rerun.yearly_states[year].data["vintage_vehicle"]
    match = orig_vs.age_distribution == rerun_vs.age_distribution
    all_match = all_match and match
    summary = VintageSummary.from_state(orig_vs)
    symbol = "✓" if match else "✗"
    print(f"  {year}: {symbol} total={summary.total_count}, cohorts={summary.cohort_count}")

if all_match:
    print("\n✓ Vintage state is fully deterministic across all years")

### Export simulation results for external analysis

ReformLab provides comprehensive export capabilities: panel outputs (CSV/Parquet), indicator tables, and comparison tables. All exports preserve provenance metadata.

In [ ]:
import tempfile

# Create export directory
export_dir = Path(tempfile.mkdtemp())
print(f"Export directory: {export_dir}\n")

# Export reform scenario panel to Parquet
panel_parquet = result_vintage.export_parquet(export_dir / "reform_panel.parquet")
print(f"✓ Panel exported to Parquet: {panel_parquet}")
print(f"  Size: {panel_parquet.stat().st_size:,} bytes")
print(f"  Rows: {result_vintage.panel_output.table.num_rows:,}")
years = sorted(result_vintage.yearly_states.keys())
print(f"  Years: {years[0]}-{years[-1]}")
print()

# Verify Parquet metadata includes provenance
import pyarrow.parquet as pq
loaded_panel = pq.read_table(panel_parquet)
schema_metadata = loaded_panel.schema.metadata
if schema_metadata and b"reformlab_panel_version" in schema_metadata:
    print(f"✓ Parquet schema includes provenance metadata")
    print(f"  Panel version: {schema_metadata[b'reformlab_panel_version'].decode()}")
print()

In [ ]:
# Export indicator and comparison tables
fiscal_parquet = fiscal_reform.export_parquet(export_dir / "fiscal_indicators.parquet")
print(f"✓ Fiscal indicators exported to Parquet: {fiscal_parquet}")
print(f"  Size: {fiscal_parquet.stat().st_size:,} bytes")
print()

comparison_parquet = fiscal_comparison.export_parquet(export_dir / "fiscal_comparison.parquet")
print(f"✓ Comparison table exported to Parquet: {comparison_parquet}")
print(f"  Size: {comparison_parquet.stat().st_size:,} bytes")
print(f"  Scenarios compared: baseline vs. reform")
print()

# Verify round-trip
reloaded_comparison = pq.read_table(comparison_parquet)
print(f"✓ Parquet round-trip verification:")
print(f"  Original rows: {fiscal_comparison.table.num_rows}")
print(f"  Reloaded rows: {reloaded_comparison.num_rows}")
print(f"  Schema preserved: {reloaded_comparison.schema == fiscal_comparison.table.schema}")
print(f"  Columns: {', '.join(reloaded_comparison.column_names)}")

---
## Section 6: Next Steps

You've now mastered ReformLab's advanced features! Here's what to explore next:

### What You've Learned
- **Typed policy objects** (`CarbonTaxParameters`, custom `SimpleFeebateParameters`) replace raw parameter dicts
- **Multi-year orchestration** with escalating rate schedules and vintage tracking
- **Scenario comparison** with distributional and fiscal indicators
- **Custom policy types** — subclass `PolicyParameters` to model any policy
- **Lineage and reproducibility** via immutable run manifests

### Advanced Topics
1. **Behavioral response steps**: Vintage fleet composition feeding back into computation (Phase 2)
2. **Additional asset classes**: Heating systems, appliances — the vintage subsystem supports any `asset_class`
3. **Custom orchestrator steps**: Implement `OrchestratorStep` for your own pipeline logic
4. **Revenue-neutral reforms**: Combine carbon tax with lump-sum rebates or progressive dividends
5. **Sensitivity analysis**: Run parameter sweeps to test robustness

### The Pattern
Every ReformLab analysis follows the same flow:
1. **Subclass** `PolicyParameters` (or use a built-in type)
2. **Instantiate** with your parameters
3. **Wrap** in a `BaselineScenario`
4. **Bridge** to `ScenarioConfig` → `RunConfig`
5. **Run** and analyze

---

**You're now ready to run production-grade environmental policy analysis with ReformLab!**